In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_csv_df = spark.read.csv("data/retail_db_orders_part-00000", inferSchema=True, header=True)

In [3]:
orders_csv_df.show()

+---+---------------------+-----+---------------+
|  1|2013-07-25 00:00:00.0|11599|         CLOSED|
+---+---------------------+-----+---------------+
|  2| 2013-07-25 00:00:...|  256|PENDING_PAYMENT|
|  3| 2013-07-25 00:00:...|12111|       COMPLETE|
|  4| 2013-07-25 00:00:...| 8827|         CLOSED|
|  5| 2013-07-25 00:00:...|11318|       COMPLETE|
|  6| 2013-07-25 00:00:...| 7130|       COMPLETE|
|  7| 2013-07-25 00:00:...| 4530|       COMPLETE|
|  8| 2013-07-25 00:00:...| 2911|     PROCESSING|
|  9| 2013-07-25 00:00:...| 5657|PENDING_PAYMENT|
| 10| 2013-07-25 00:00:...| 5648|PENDING_PAYMENT|
| 11| 2013-07-25 00:00:...|  918| PAYMENT_REVIEW|
| 12| 2013-07-25 00:00:...| 1837|         CLOSED|
| 13| 2013-07-25 00:00:...| 9149|PENDING_PAYMENT|
| 14| 2013-07-25 00:00:...| 9842|     PROCESSING|
| 15| 2013-07-25 00:00:...| 2568|       COMPLETE|
| 16| 2013-07-25 00:00:...| 7276|PENDING_PAYMENT|
| 17| 2013-07-25 00:00:...| 2667|       COMPLETE|
| 18| 2013-07-25 00:00:...| 1205|         CLOSED|


In [4]:
header_col = ['order_id', 'order_date', 'customer_id', 'customer_status']

In [5]:
orders_df = orders_csv_df.toDF(*header_col)

In [6]:
orders_df.show()

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|customer_status|
+--------+--------------------+-----------+---------------+
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
|       5|2013-07-25 00:00:...|      11318|       COMPLETE|
|       6|2013-07-25 00:00:...|       7130|       COMPLETE|
|       7|2013-07-25 00:00:...|       4530|       COMPLETE|
|       8|2013-07-25 00:00:...|       2911|     PROCESSING|
|       9|2013-07-25 00:00:...|       5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:...|       5648|PENDING_PAYMENT|
|      11|2013-07-25 00:00:...|        918| PAYMENT_REVIEW|
|      12|2013-07-25 00:00:...|       1837|         CLOSED|
|      13|2013-07-25 00:00:...|       9149|PENDING_PAYMENT|
|      14|2013-07-25 00:00:...|       9842|     PROCESSING|
|      15|2013-07-25 00:00:...|       25

### 1st row is gone. Above due to header=true. Let's start over

In [7]:
orders_new_df = spark.read.csv("data/retail_db_orders_part-00000", inferSchema=True)

In [8]:
orders_new_df.show()

+---+--------------------+-----+---------------+
|_c0|                 _c1|  _c2|            _c3|
+---+--------------------+-----+---------------+
|  1|2013-07-25 00:00:...|11599|         CLOSED|
|  2|2013-07-25 00:00:...|  256|PENDING_PAYMENT|
|  3|2013-07-25 00:00:...|12111|       COMPLETE|
|  4|2013-07-25 00:00:...| 8827|         CLOSED|
|  5|2013-07-25 00:00:...|11318|       COMPLETE|
|  6|2013-07-25 00:00:...| 7130|       COMPLETE|
|  7|2013-07-25 00:00:...| 4530|       COMPLETE|
|  8|2013-07-25 00:00:...| 2911|     PROCESSING|
|  9|2013-07-25 00:00:...| 5657|PENDING_PAYMENT|
| 10|2013-07-25 00:00:...| 5648|PENDING_PAYMENT|
| 11|2013-07-25 00:00:...|  918| PAYMENT_REVIEW|
| 12|2013-07-25 00:00:...| 1837|         CLOSED|
| 13|2013-07-25 00:00:...| 9149|PENDING_PAYMENT|
| 14|2013-07-25 00:00:...| 9842|     PROCESSING|
| 15|2013-07-25 00:00:...| 2568|       COMPLETE|
| 16|2013-07-25 00:00:...| 7276|PENDING_PAYMENT|
| 17|2013-07-25 00:00:...| 2667|       COMPLETE|
| 18|2013-07-25 00:0

In [9]:
from pyspark.sql.functions import col
order_renamed_df = orders_new_df.select(
    col('_c0').alias('order_id'),
    col('_c1').alias('order_date'),
    col('_c2').alias('customer_id'),
    col('_c3').alias('order_status')
)

In [10]:
order_renamed_df.show()

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
|       5|2013-07-25 00:00:...|      11318|       COMPLETE|
|       6|2013-07-25 00:00:...|       7130|       COMPLETE|
|       7|2013-07-25 00:00:...|       4530|       COMPLETE|
|       8|2013-07-25 00:00:...|       2911|     PROCESSING|
|       9|2013-07-25 00:00:...|       5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:...|       5648|PENDING_PAYMENT|
|      11|2013-07-25 00:00:...|        918| PAYMENT_REVIEW|
|      12|2013-07-25 00:00:...|       1837|         CLOSED|
|      13|2013-07-25 00:00:...|       9149|PENDING_PAYMENT|
|      14|2013-07-25 00:00:...|       98

In [11]:
order_renamed_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [12]:
order_renamed_df.dtypes

[('order_id', 'int'),
 ('order_date', 'string'),
 ('customer_id', 'int'),
 ('order_status', 'string')]

In [13]:
order_renamed_df.createOrReplaceTempView('orders_table')

In [14]:
spark.sql("select * from orders_table limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
2,2013-07-25 00:00:...,256,PENDING_PAYMENT
3,2013-07-25 00:00:...,12111,COMPLETE
4,2013-07-25 00:00:...,8827,CLOSED
5,2013-07-25 00:00:...,11318,COMPLETE


In [15]:
spark.sql("describe extended order_renamed_df")

AnalysisException: Table or view not found for 'DESCRIBE TABLE': order_renamed_df; line 1 pos 0;
'DescribeRelation true, [col_name#194, data_type#195, comment#196]
+- 'UnresolvedTableOrView [order_renamed_df], DESCRIBE TABLE, true


In [16]:
spark.sql("show tables")

database,tableName,isTemporary
default,1htab,false
default,41group_movies,false
default,4group_movies,false
default,4tab,false
default,6_flags_simon,false
default,a,false
default,aa,false
default,abc,false
default,acid,false
default,acid1,false


In [17]:
spark.sql("use ecomm")

""


In [18]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS ecomm.external_orders
    USING PARQUET
    LOCATION 'data/external_table'
    AS SELECT * FROM orders_table
""")

""


In [19]:
spark.sql("select * from ecomm.external_orders limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
2,2013-07-25 00:00:...,256,PENDING_PAYMENT
3,2013-07-25 00:00:...,12111,COMPLETE
4,2013-07-25 00:00:...,8827,CLOSED
5,2013-07-25 00:00:...,11318,COMPLETE


In [20]:
spark.sql("SELECT * FROM orders_table limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:...,11599,CLOSED
2,2013-07-25 00:00:...,256,PENDING_PAYMENT
3,2013-07-25 00:00:...,12111,COMPLETE
4,2013-07-25 00:00:...,8827,CLOSED
5,2013-07-25 00:00:...,11318,COMPLETE


In [21]:
spark.sql("DESCRIBE FORMATTED ecomm.external_orders").show(100, truncate=False)

+----------------------------+----------------------------------------------------------------+-------+
|col_name                    |data_type                                                       |comment|
+----------------------------+----------------------------------------------------------------+-------+
|order_id                    |int                                                             |null   |
|order_date                  |string                                                          |null   |
|customer_id                 |int                                                             |null   |
|order_status                |string                                                          |null   |
|                            |                                                                |       |
|# Detailed Table Information|                                                                |       |
|Database                    |ecomm                             

#### I see 'The Disconnect'

    files are at: /user/itv025320/data/external_table

    The Spark is looking: /user/itv025320/warehouse/ecomm.db/data/external_table
    
### So I'm gonna point the spark to the correct location since I had previously set the metastore at hive.metastore.warehouse.dir=/user/{username}/warehouse;

## Spark took relative string and appended it to the Database's root directory in the warehouse.

In [22]:
spark.sql("drop table ecomm.external_orders")

""


In [23]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS ecomm.external_orders
    using parquet
    location '/user/itv025320/data/external_table'
    as select * from orders_table
""")

""


In [24]:
spark.sql("select * from ecomm.external_orders limit 5").show(truncate=False)

+--------+---------------------+-----------+---------------+
|order_id|order_date           |customer_id|order_status   |
+--------+---------------------+-----------+---------------+
|1       |2013-07-25 00:00:00.0|11599      |CLOSED         |
|2       |2013-07-25 00:00:00.0|256        |PENDING_PAYMENT|
|3       |2013-07-25 00:00:00.0|12111      |COMPLETE       |
|4       |2013-07-25 00:00:00.0|8827       |CLOSED         |
|5       |2013-07-25 00:00:00.0|11318      |COMPLETE       |
+--------+---------------------+-----------+---------------+



In [25]:
spark.sql("describe extended ecomm.external_orders").show(truncate=False)

+----------------------------+----------------------------------------------------------------+-------+
|col_name                    |data_type                                                       |comment|
+----------------------------+----------------------------------------------------------------+-------+
|order_id                    |int                                                             |null   |
|order_date                  |string                                                          |null   |
|customer_id                 |int                                                             |null   |
|order_status                |string                                                          |null   |
|                            |                                                                |       |
|# Detailed Table Information|                                                                |       |
|Database                    |ecomm                             

order_date    is of data type              **string**
So I am thinking of changing the data type to **timestamp**. Let's see...

If I simply do ALTER TABLE command to change the type—Spark/Hive usually won't allow a "casting" change in the metadata if it might break the read.
Changing a column's data type in a Spark external table is a two-step process.
1. Update the Metadata (how Spark "sees" the data) and 
2. ensure the Physical Data (the Parquet files) actually contains timestamp-compatible information.



### Approach 1. Overwrite the table:
#### Casting the column in the SELECT statement

```
spark.sql("""
    CREATE OR REPLACE TABLE ecomm.external_orders
    USING PARQUET
    LOCATION '/user/{username}/data/external_table'
    AS SELECT
        order_id,
        to_timestamp(order_date) as order_date,
        customer_id,
        order_status
    FROM orders_table
""")
```

### Approach 2.The Metadata-Only Approach:
#### If the data in the column already follows standard time stamp format then just change schema in the metastore

```
ALTER TABLE ecomm.external_orders CHANGE COLUMN order_date order_date TIMESTAMP;
```

In [26]:
# Let's try approach 2 out
spark.sql("""
ALTER TABLE ecomm.external_orders CHANGE COLUMN order_date order_date TIMESTAMP
""")

AnalysisException: ALTER TABLE CHANGE COLUMN is not supported for changing column 'order_date' with type 'StringType' to 'order_date' with type 'TimestampType'

In [27]:
# Now let's try approach 1
spark.sql("""
    CREATE OR REPLACE TABLE ecomm.external_orders
    USING PARQUET
    LOCATION '/user/{username}/data/external_table'
    AS SELECT
        order_id,
        to_timestamp(order_date) as order_date,
        customer_id,
        order_status
    FROM orders_table
""")

AnalysisException: REPLACE TABLE AS SELECT is only supported with v2 tables.

In [28]:
# Okay. Not supported. I'll go step by step then
spark.sql("DROP TABLE IF EXISTS ecomm.external_orders")

""


In [30]:
spark.sql("""
    CREATE TABLE ecomm.external_orders 
    USING PARQUET
    LOCATION '/user/itv025320/data/external_table'
    AS SELECT
        order_id,
        cast(order_date as timestamp) as order_date,
        customer_id,
        order_status
    FROM orders_table
""")

""


In [31]:
spark.sql("select * from external_orders limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:00,11599,CLOSED
2,2013-07-25 00:00:00,256,PENDING_PAYMENT
3,2013-07-25 00:00:00,12111,COMPLETE
4,2013-07-25 00:00:00,8827,CLOSED
5,2013-07-25 00:00:00,11318,COMPLETE


In [33]:
spark.table("ecomm.external_orders").printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [34]:
spark.stop()